In [2]:
from ultralytics import YOLO

model = YOLO("models/base/yolo26n-obb.pt") 

file:models/base/yolo26n-obb.pt


In [ ]:
import albumentations as A

IMG_SIZE = 720

# Aggressive augmentations layered on top of Ultralytics' built-in geometric/color jitter.
# Targets robustness to phone-camera artefacts: tight/partial crops, sensor noise, JPEG compression.
augmentations = [
    # A.RandomSizedBBoxSafeCrop(height=IMG_SIZE, width=IMG_SIZE, erosion_rate=0.2, p=0.3),
    # A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(0.05, 0.2), hole_width_range=(0.05, 0.2), p=0.3),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
    A.ISONoise(p=0.2),
    A.ImageCompression(quality_range=(40, 90), p=0.3),
    A.Blur(blur_limit=5, p=0.1),
    A.MedianBlur(blur_limit=5, p=0.05),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.2),
    A.ToGray(p=0.02),
    A.CLAHE(p=0.05),
]

model.train(data="./datasets/SKU110K_fixed/sku110k_obb.yaml", 
    epochs=150, 
    imgsz=IMG_SIZE,
    single_cls=True,
    degrees=10.0,    # Slightly rotate images
    flipud=0.5,      # Flip up-down
    fliplr=0.5,      # Flip left-right
    mosaic=1.0,       # Combine multiple images into one
    scale=0.9,
    shear=10.0,
    perspective=0.0005,
    mixup=0.15,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    batch=16,
    augmentations=augmentations,
    )

In [ ]:
model.save("./models/tuned/items_detect.yolo26n-obb.pt")

In [ ]:
import glob
import os
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO

model = YOLO("./models/tuned/items_detect.yolo26n-obb.pt")

print("loaded model")
test_images_path = "./datasets/SKU110K_fixed/images/test/*.jpg"  # Update path if needed
image_paths = glob.glob(test_images_path)

if not image_paths:
    raise FileNotFoundError("No images found. Please verify your test images path.")

# Select a random subset to display (e.g., 4 samples)
num_samples = min(4, len(image_paths))
sampled_paths = random.sample(image_paths, num_samples)

# 3. Setup Matplotlib grid (assuming a 2x2 grid for 4 samples)
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.ravel()  # Flatten the 2D array of axes into 1D for easy iteration

# 4. Infer and plot
for idx, img_path in enumerate(sampled_paths):
    # Run segmentation prediction
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    
    # Extract the first result from the list
    result = results[0]
    

    annotated_img = result.plot(pil=True, boxes=True, masks=True)
    
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(os.path.basename(img_path), fontsize=10)
    axes[idx].axis("off")  # Hide grid borders and coordinates

# Tight layout adjustments for notebook embedding
plt.tight_layout()
plt.show()